In [1]:
// load the formatting setup and the style sheet
#load "../setup.fsx"
HTML(System.IO.File.ReadAllText "../style.css")

The below script needs to be able to find the current output cell; this is an easy method to get it.

# Integer Square Root

In this notebook we work with numbers that are bigger than $2^{64}$.  Therefore, we use the type `bigint`, which represents integers of arbitrary size.  Literals of type `bigint` are written with the suffix `I`, e.g. `0I`.  The function `pown x k` computes the power $x^k$ where `k` is an `int`.

The function `isqrt` takes one argument:
- `a` is a natural number that is less than $2^{128}$.

It returns the largest natural number $r$ such that
$r^2 \leq a$, i.e. we have
$$ \texttt{isqrt}(a) := \max\bigl(\{ r \in \mathbb{N} \mid r^2 \leq a \}\bigr). $$

**Implementation:** Since $a < 2^{128}$ we know that `isqrt a` is less than $2^{64}$.  Hence the result has at most $64$ bits.
The implementation computes the result bit by bit starting from the bit with the highest value $2^{63}$.  The mutable variable `p` runs through the powers $2^{63}, 2^{62}, \cdots, 2^1, 2^0$.  For every `p` we check whether we can add `p` to `result` without the square of `result` exceeding `a`.  If this is the case, the bit `p` is set in `result`.

If `a` is not less than $2^{128}$, an exception is raised.

In [2]:
let isqrt (a: bigint) : bigint =
    if a >= pown 2I 128 then
        failwith "isqrt: argument too big"
    let mutable result = 0I
    let mutable p      = pown 2I 63
    while p > 0I do
        if a >= pown (result + p) 2 then
            result <- result + p
        p <- p / 2I
    result

In [3]:
for n in 0 .. 9 do
    printfn "isqrt(%d) = %A" n (isqrt (bigint n))

isqrt(

0

) = 

0

isqrt(

...

2

isqrt(

9

) = 

3

In order to test our implementation more thoroughly we use random numbers.  We create a random number generator with the seed `0`, so that the tests are reproducible.

In [4]:
let rnd = System.Random(0)

The function `randomBigint` takes one argument:
- `bits` is a natural number that is a multiple of $8$.

It returns a random natural number $n$ such that $0 \leq n < 2^{\texttt{bits}}$.

**Implementation:** We fill an array of `bits / 8` bytes with random values.  These bytes are then interpreted as the binary representation of an unsigned integer.

In [5]:
let randomBigint (bits: int) : bigint =
    let bytes = Array.zeroCreate<byte> (bits / 8)
    rnd.NextBytes bytes
    bigint(System.ReadOnlySpan bytes, isUnsigned = true)

The function `runTests` takes two arguments:
- `noTests` is the number of tests that are to be performed,
- `f` is a function that takes a natural number $n < 2^{128}$ and is supposed to compute the integer square root of $n$.

The function generates `noTests` random integers `n` less than $2^{128}$ and tests, whether 
`f n` is the *integer square root* of `n` in each case.  If a test fails, an exception is raised.

In [6]:
let runTests (noTests: int) (f: bigint -> bigint) : unit =
    for _ in 1 .. noTests do
        let n = randomBigint 128
        let r = f n
        if not (r * r <= n && pown (r + 1I) 2 > n) then
            failwithf "Error: %A != f(%A)" r n

In [7]:
#!time
runTests 100_000 isqrt

Wall time: 1185.4223ms

With 128 bits we can approximate the square root of $2$ up to $18$ decimal places.

In [8]:
2I * pown 10I 38 < pown 2I 128

true

In [9]:
let s = string (isqrt (2I * pown 10I 38))
printfn "%s.%s" (s.Substring(0, 1)) (s.Substring 1)

1

.

4142135623730950488

We can check this at [CATONMAT](https://catonmat.net/tools/generate-sqrt2-digits).

When we compare this with the floating point implementation of the function `sqrt`, we can see that the floating point implementation has been rounded up in the last digit.

In [10]:
sqrt 2.0

1.4142135623730951

1.4142135623730950488